# Aurum: XAUUSD işlem günlüğü analizi

**KURGUSAL VERİ — Gerçek performans veya strateji etkinliği kanıtı değildir.**

Araştırma sorusu: Plana uyulan ve uyulmayan işlemler arasında ortalama net sonuç farkı var mı?

Bu notebook veri doğrulama, eksik risk analizi, Pandas gruplama, SQL ve keşif amaçlı bootstrap içerir. Küçük ve kurgusal örneklem yalnızca yöntemi gösterir. Hesap para birimi USD olarak varsayılmıştır; kur dönüşümü yapılmaz.

In [1]:
import io, csv, sqlite3, random, json
import pandas as pd
from statistics import mean
CSV = 'date,symbol,side,net_pnl,strategy,followed_plan,time,risk_amount,entry,exit,stop,lot,tick_size,tick_value,gross_pnl,commission,swap,fee,note\n2026-09-01,XAUUSD,long,125,Breakout,yes,10:15:00,100,,,,,,,130.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-02,XAUUSD,short,-80,Breakout,no,15:40:00,100,,,,,,,-75.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-03,XAUUSD,long,180,Pullback,yes,16:20:00,100,,,,,,,185.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-04,XAUUSD,long,-60,Pullback,yes,10:15:00,100,,,,,,,-55.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-07,XAUUSD,short,-150,Breakout,no,15:40:00,100,,,,,,,-145.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-08,XAUUSD,long,210,Pullback,yes,16:20:00,100,,,,,,,215.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-09,XAUUSD,short,95,Breakout,yes,10:15:00,100,,,,,,,100.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-10,XAUUSD,long,0,Pullback,yes,15:40:00,100,,,,,,,5.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-11,XAUUSD,short,-110,Breakout,no,16:20:00,100,,,,,,,-105.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-14,XAUUSD,long,165,Pullback,yes,10:15:00,100,,,,,,,170.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-15,XAUUSD,long,-70,Pullback,yes,15:40:00,,,,,,,,-65.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n2026-09-16,XAUUSD,short,140,Breakout,yes,16:20:00,,,,,,,,145.0,-4,-1,0,Kurgusal eğitim kaydı; gerçek fiyat veya işlem değildir.\n'
df = pd.read_csv(io.StringIO(CSV))
print(df[['date','side','net_pnl','strategy','followed_plan']].to_string(index=False))

      date  side  net_pnl strategy followed_plan
2026-09-01  long      125 Breakout           yes
2026-09-02 short      -80 Breakout            no
2026-09-03  long      180 Pullback           yes
2026-09-04  long      -60 Pullback           yes
2026-09-07 short     -150 Breakout            no
2026-09-08  long      210 Pullback           yes
2026-09-09 short       95 Breakout           yes
2026-09-10  long        0 Pullback           yes
2026-09-11 short     -110 Breakout            no
2026-09-14  long      165 Pullback           yes
2026-09-15  long      -70 Pullback           yes
2026-09-16 short      140 Breakout           yes


## Veri kalitesi
Eksik riski sıfır saymayız. Aynı değerli işlemler olası tekrar olarak incelenir; otomatik silinmez. Komisyon, swap ve fee işaretli nakit akışıdır.

In [2]:
assert pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce').notna().all()
assert df['side'].isin(['long','short']).all()
assert df['followed_plan'].isin(['yes','no','unknown']).all()
assert (abs(df.net_pnl-(df.gross_pnl+df.commission+df.swap+df.fee))<.011).all()
print('Olası tekrar:', df.duplicated(subset=['date','time','symbol','side','net_pnl','entry','exit','lot']).sum())
print('Eksik risk:', df.risk_amount.isna().sum())
assert len(df)==12 and df.net_pnl.sum()==445


Olası tekrar: 0
Eksik risk: 2


## Bakiye ve risk
Kapanış sırası kullanılır. Başlangıç bakiyesi 10.000 USD. R = net sonuç / başlangıç fiyat riski; risk masraf ve kayma içermez.

In [3]:
df = df.sort_values(['date','time'], kind='stable').reset_index(drop=True)
df['equity'] = 10000 + df.net_pnl.cumsum()
df['peak'] = df.equity.cummax().clip(lower=10000)
df['drawdown'] = df.peak-df.equity
df['r_multiple'] = df.net_pnl/df.risk_amount.where(df.risk_amount>0)
print('Net:', df.net_pnl.sum(), 'Maksimum düşüş:', df.drawdown.max())
print('Risk kapsamı:', df.r_multiple.notna().sum(), '/', len(df))
print(df.groupby('strategy').agg(n=('net_pnl','size'),net=('net_pnl','sum'),average=('net_pnl','mean'),average_r=('r_multiple','mean'),r_n=('r_multiple','count')).to_string())
assert df.drawdown.max()==210


Net: 445 Maksimum düşüş: 210
Risk kapsamı: 10 / 12
          n  net    average  average_r  r_n
strategy                                   
Breakout  6   20   3.333333      -0.24    5
Pullback  6  425  70.833333       0.99    5


## SQL ile bağımsız kontrol
SQL analitik bir kopyada çalışır; uygulamanın tarayıcı kayıtlarından bağımsızdır.

In [4]:
con=sqlite3.connect(':memory:')
df.to_sql('trades',con,index=False)
query='SELECT strategy, COUNT(*) AS n, SUM(net_pnl) AS net, AVG(net_pnl) AS average FROM trades GROUP BY strategy'
print(pd.read_sql_query(query,con).to_string(index=False))
assert con.execute('SELECT SUM(net_pnl) FROM trades').fetchone()[0]==445
con.close()


strategy  n  net   average
Breakout  6   20  3.333333
Pullback  6  425 70.833333


## Belirsizlik: bootstrap
Uyulan − uyulmayan ortalama net farkı için 3.000 yeniden örnekleme. İşlemleri bağımsız kabul eden keşif amaçlı yöntemdir; seri bağımlılığını veya işlem büyüklüğünü kontrol etmez.

In [5]:
yes=df.loc[df.followed_plan=='yes','net_pnl'].tolist()
no=df.loc[df.followed_plan=='no','net_pnl'].tolist()
rng=random.Random(42)
boot=sorted(mean(rng.choices(yes,k=len(yes)))-mean(rng.choices(no,k=len(no))) for _ in range(3000))
print('Grup sayıları:',len(yes),len(no))
print('Ortalama fark:',round(mean(yes)-mean(no),2))
print('Keşif amaçlı %95 aralık:',round(boot[75],2),round(boot[2925],2))


Grup sayıları: 9 3
Ortalama fark: 200.56
Keşif amaçlı %95 aralık: 124.44 271.11


## Yorum ve sınırlar

Bu örnekteki farklar kurgusaldır. Plan dışı işlemlerin daha kötü olması gerçek hayata genellenemez. Plan etiketi işlemden sonra verildiyse sonuç yanlılığı oluşabilir. Gelecek çalışmada etiketi işlem öncesinde belirleme, gerçek veriyle daha uzun dönem toplama, strateji ve risk büyüklüğüne göre tabakalama, gün bazında block bootstrap ve zaman bazlı doğrulama gerekir.

Kapanış bakiyesi açık pozisyon riskini göstermez. Hesap hareketleri bu veri sözleşmesinde yoktur. Kısmi kapanışlar ve tersine dönüşlü ham MT5 deal kayıtları pozisyonlara dönüştürülmeden analiz edilmemelidir.

Kaynak: Projede oluşturulan 12 sentetik işlem; gerçek piyasa fiyatı içermez. MT5 ürün özellikleri: https://www.metatrader5.com/en/terminal/help/trading/market_watch
